In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install fastapi uvicorn pyngrok transformers==4.52.4 accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 74.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.2 MB/s eta 0:00:00:00:01


In [3]:
!pip install youtube-transcript-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 6.7 MB/s eta 0:00:00:00:010:01


In [4]:
from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi

In [5]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=0
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [6]:
def extract_video_id(url: str) -> str:
    parsed = urlparse(url)
    host = (parsed.hostname or "").lower()

    if host.startswith("www."):
        host = host[4:]

    if host == "youtu.be":
        video_id = parsed.path.lstrip("/").split("/")[0]

        if video_id:
            return video_id

    if host in ("youtube.com", "m.youtube.com", "music.youtube.com"):
        qs = parse_qs(parsed.query)

        if qs.get("v"):
            return qs["v"][0]

        parts = parsed.path.strip("/").split("/")

        if len(parts) >= 2 and parts[0] in (
            "shorts",
            "embed",
            "live",
            "v"
        ):
            return parts[1]

    raise ValueError(f"No video id found in URL: {url}")

In [7]:
def get_transcript(url: str) -> str:
    video_id = extract_video_id(url)

    api = YouTubeTranscriptApi()

    fetched = api.fetch(
        video_id,
        languages=["ar", "en"]
    )

    text = "\n".join(
        snippet.text
        for snippet in fetched
    )

    return text

In [8]:
def summarize_text(text: str) -> str:

    tokens = summarizer.tokenizer(
        text,
        return_tensors="pt",
        truncation=False
    )

    input_ids = tokens["input_ids"][0]

    chunk_size = 800

    chunks = []

    for i in range(0, len(input_ids), chunk_size):

        chunk_ids = input_ids[i:i + chunk_size]

        chunk_text = summarizer.tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True
        )

        chunks.append(chunk_text)

    summaries = []

    for chunk in chunks:

        result = summarizer(
            chunk,
            max_length=180,
            min_length=60,
            do_sample=False
        )

        summaries.append(
            result[0]["summary_text"]
        )

    combined_summary = " ".join(summaries)

    final_result = summarizer(
        combined_summary,
        max_length=250,
        min_length=100,
        do_sample=False
    )

    return final_result[0]["summary_text"]

In [9]:
url = "https://www.youtube.com/watch?v=NbUTIFEEXLY"

transcript = get_transcript(url)

print("Transcript length:", len(transcript))

summary = summarize_text(transcript)

print("\nFINAL SUMMARY:\n")
print(summary)

Transcript length: 6646


Your max_length is set to 180, but your input_length is only 157. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=78)
Your max_length is set to 250, but your input_length is only 232. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=116)



FINAL SUMMARY:

GBD-6 Astra is a new computer use model. It has been compared to GPT-4 Soul, which went beyond the authorized target 48% of the time. Astra scored a 92% on ScreenSpot Pro, which is a benchmark for computer use. The pricing for Astra seems to be double that of 5.6 Soul. I'm really excited to see when we get it. It says, you know, over                the next coming days. Hopefully, we get that very, very soon.


In [10]:
API_KEY = "secret123"

In [11]:
from fastapi import FastAPI, Request, HTTPException
from pydantic import BaseModel

app = FastAPI()

class VideoRequest(BaseModel):
    url: str


@app.post("/summarize")
async def summarize_video(
    request: Request,
    data: VideoRequest
):
    if request.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(
            status_code=401,
            detail="Unauthorized"
        )

    try:
        transcript = get_transcript(data.url)
        summary = summarize_text(transcript)

        return {
            "summary": summary
        }

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )

In [12]:
from pyngrok import ngrok, conf

NGROK_TOKEN = "3K8cjopq4Knddy3CBoS6IsJMFpW_zV8h4E6qLnTNjQGwYKZK"

conf.get_default().auth_token = NGROK_TOKEN

In [13]:
import socket

def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()

print("Port:", port)

Port: 57889


In [14]:
public_url = ngrok.connect(port).public_url

print("Your public URL:", public_url)

Your public URL: https://huskiness-antiques-theatrics.ngrok-free.dev                                


In [15]:
import uvicorn
import threading
import time

def run():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=port
    )

threading.Thread(target=run, daemon=True).start()

time.sleep(2)

print("FastAPI is running on port:", port)
print("Public URL:", public_url)

INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:57889 (Press CTRL+C to quit)


FastAPI is running on port: 57889
Public URL: https://huskiness-antiques-theatrics.ngrok-free.dev


In [16]:
import requests

response = requests.post(
    f"{public_url}/summarize",
    headers={
        "Authorization": f"Bearer {API_KEY}"
    },
    json={
        "url": "https://www.youtube.com/watch?v=NbUTIFEEXLY"
    }
)

print("Status:", response.status_code)
print(response.json())

Your max_length is set to 180, but your input_length is only 157. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=78)
Your max_length is set to 250, but your input_length is only 232. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=116)


INFO:     34.66.41.68:0 - "POST /summarize HTTP/1.1" 200 OK
Status: 200
{'summary': "GBD-6 Astra is a new computer use model. It has been compared to GPT-4 Soul, which went beyond the authorized target 48% of the time. Astra scored a 92% on ScreenSpot Pro, which is a benchmark for computer use. The pricing for Astra seems to be double that of 5.6 Soul. I'm really excited to see when we get it. It says, you know, over \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0the next coming days. Hopefully, we get that very, very soon."}
